In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
import sys

#math and array operations
import numpy as np
import math
import pandas as pd

#data classes
import xarray as xr
import pickle

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import matplotlib.gridspec as gridspec

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#loading bar
from tqdm import tqdm

#datetime
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
# dataType = "RadarComparison"
dataType = "RadarComparison_Interpolation" #*INTERPOLATION

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12";# spinup_hours="-16"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#Importing ERA5 Data Loading Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","ERA5_Data"))
from CLASSES_ERA5DataLoading import ERA5DataLoading_Class,ERA5DataLoading_Class_gdex

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData_NSSL.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetLoopElements(start_job,end_job):
    loop_elements = np.arange(ModelData_NSSL.Ntime)[start_job:end_job].tolist()
    return loop_elements
loop_elements = GetLoopElements(start_job,end_job)

In [ ]:
########################
#DATA INFORMATION

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. X-Band Scanning ARM Cloud Radar (XSACRCFRQC), 2022-06-09 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by Y. Feng, A. Matthews, E. Schuman, K. Johnson, I. Lindenmaier, V. Castro and T. Wendler. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/2001296.

#Globus Download Link
# https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261781*__;Ly8v!!PvDODwlR4mBZyAb0!REKAOzHJNvJk50CY5Pjl135CV83BhArwtdyMDuBM-28KqreBug8Xb5Mc3MLgy_p9PUiWe2uVXq-EfUtLcGdH5w$

In [ ]:
#DATA CITATION
# Atmospheric Radiation Measurement (ARM) user facility. 2021. Ka-Band Scanning ARM Cloud Radar (KASACRCFRQC), 2022-06-08 to 2022-07-02, ARM Mobile Facility (HOU) Houston, TX; AMF1 (main site for TRACER) (M1). Compiled by I. Lindenmaier, K. Johnson, D. Nelson, A. Matthews, T. Wendler, V. Melo de Castro, M. Rocque and Y. Feng. ARM Data Center. Data set accessed 2025-11-05 at http://dx.doi.org/10.5439/1877338.

# https://armgov.svcs.arm.gov/capabilities/instruments/kasacr

#Globus Download Link
#https://urldefense.com/v3/__https://app.globus.org/file-manager?origin_id=ba87aabe-30f6-433d-b4a5-19434c595e0f&origin_path=*rosemana1*261893*__;Ly8v!!PvDODwlR4mBZyAb0!UJrKWg_xaYuaaa_iaY91TCVMB_neSskXsDKHtPPCG7Ix6sEmiEvnngTUrYuV18LudZqxB3eHyTDuCHSZ3o3zrg$

In [ ]:
#LOADING RADAR CLASS
if int(ModelData_NSSL.spinup_hours) <= 0:
    if ModelData_NSSL.region == "TRACER":
        if ModelData_NSSL.case == "WET":
            dateString = '2022-06-30_2022-07-03'
        elif ModelData_NSSL.case == "DIURNAL":
            dateString = '2022-06-21_2022-06-24'
    elif ModelData_NSSL.region == "Hawaii":
        if ModelData_NSSL.case == "WET":
            dateString = "2021-12-05_2021-12-08"
        elif ModelData_NSSL.case == "TRADES":
            dateString = "2022-08-07_2022-08-10"
        
else:
    dateString = f"{ModelData_NSSL.simulationDates[0]}_{ModelData_NSSL.simulationDates[-1]}"

RadarData_MRMS = RadarData_MRMS_Class(ModelData_NSSL,
                                      fileDirectory=os.path.join(DirectoryManager.dataDirectory,
                                                                 f"Observation_Data/{ModelData_NSSL.region}/MRMS_RadarData",
                                                                 dateString,
                                                                 "MergedReflectivityQC_01.00"))

In [ ]:
##########################
#DATA LOADING FUNCTIONS

In [ ]:
#Converting timeStrings
def ConvertTimeStringtoDateTime(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    """
    return datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
def ConvertTimeStringtoTimeTitle(timeString):
    """
    Converts a time string like '2022-06-30_00.00.00' to a datetime object.
    Formatted for use as a plot title.
    """
    # Parse the custom format to a datetime object
    dt = datetime.strptime(timeString, '%Y-%m-%d_%H.%M.%S')
    # Format it to 'YYYY-MM-DD HH:MM:SS'
    return dt.strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)

In [ ]:
def FixLatLon_RadarData(radarData):        
    radarData = radarData.isel(latitude=slice(None, None, -1))

    radarData = radarData.assign_coords(
        longitude=((radarData.longitude + 180) % 360) - 180
    )
    return radarData

def ReturnLatLon_RadarData(radarData):
    # Fix latitude order
    radarData_fixed = radarData.isel(latitude=slice(None, None, -1))

    # Fix longitude convention
    radarData_fixed = radarData_fixed.assign_coords(
        longitude=radarData.longitude+360
    )

    return radarData_fixed

def InterpolateRadarData(radarData,modelData):
    radarData = FixLatLon_RadarData(radarData)
    
    radarData_MRMS = radarData.interp(
        latitude=modelData.latitude,
        longitude=modelData.longitude,
        method="linear"
    )
    return radarData_MRMS

In [ ]:
## GetData

zGrid_f, zGrid_c = ModelData_NSSL.GetZGrids() #*INTERPOLATION
[zTarget_f, zTarget_c] = ModelData_NSSL.GetZTarget(zGrid_f, zGrid_c) #*INTERPOLATION

# z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
# zlevels = np.loadtxt(z_levels_filePath)/1e3
# zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
zlevels = zTarget_f/1e3 #*INTERPOLATION
zlevels_center = zTarget_c/1e3 #*INTERPOLATION

def GetData(t):
    timeString = ModelData_NSSL.timeStrings[t]
    timeString_datetime = ConvertTimeStringtoDateTime(timeString)
    
    #Loading Model Radar
    modelRadarData_NSSL = ModelData_NSSL.GetDataTimestep_diag(t)["refl10cm"]
    modelRadarData_TEMPO = ModelData_TEMPO.GetDataTimestep_diag(t)["refl10cm"]
    modelRadarTimeTitle = ConvertTimeStringtoTimeTitle(timeString)

    #Interpolating Z levels #*INTERPOLATION
    #################################
    modelRadarData_NSSL = ModelData_NSSL.InterpolateVertical(modelRadarData_NSSL,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
    modelRadarData_TEMPO = ModelData_TEMPO.InterpolateVertical(modelRadarData_TEMPO,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
    #################################
    
    #Loading Observational Radar
    RadarObservationLevels = RadarObservationMask_Class.LoadRadarObservationLevels_MRMS(DirectoryManager, ModelData_NSSL)
    radarData_MRMS = RadarData_MRMS.GetData_AllZLevels(DirectoryManager,ModelData_NSSL, t, RadarObservationLevels)

    # Applying RadarDataMask
    modelRadarData_NSSL = modelRadarData_NSSL.where(RadarDataMask == True)
    modelRadarData_TEMPO = modelRadarData_TEMPO.where(RadarDataMask == True)
    radarData_MRMS = radarData_MRMS.where(RadarDataMask == True)

    # Assign height to MPAS reflectivity grid
    modelRadarData_NSSL = modelRadarData_NSSL.assign_coords(
        nVertLevels=("nVertLevels", zlevels_center) 
    )
    # Assign height to MPAS reflectivity grid
    modelRadarData_TEMPO= modelRadarData_TEMPO.assign_coords(
        nVertLevels=("nVertLevels", zlevels_center) 
    )
    # Interpolate to MRMS height grid
    modelRadarData_NSSL = modelRadarData_NSSL.interp(
        nVertLevels=radarData_MRMS.heightAboveSea.data / 1e3
    )
    # Interpolate to MRMS height grid
    modelRadarData_TEMPO = modelRadarData_TEMPO.interp(
        nVertLevels=radarData_MRMS.heightAboveSea.data / 1e3
    )

    #Applying Radar Threshhold
    modelRadarData_NSSL = modelRadarData_NSSL.where(modelRadarData_NSSL>0)
    modelRadarData_TEMPO = modelRadarData_TEMPO.where(modelRadarData_TEMPO>0)
    radarData_MRMS = radarData_MRMS.where(radarData_MRMS>0)
    
    return (modelRadarData_NSSL,modelRadarData_TEMPO,radarData_MRMS)

In [ ]:
##########################
#CALCULATING FUNCTIONS

In [ ]:
## CalculateFSS_scores

# Leeuwenburg, T., Loveday, N., Ebert, E. E., Cook, H., Khanarmuei, M., Taggart, R. J., Ramanathan, N., Carroll, M., Chong, S., Griffiths, A., & Sharples, J. (2024). 
# scores: A Python package for verifying and evaluating models and predictions with xarray. Journal of Open Source Software, 9(99), 6889. https://doi.org/10.21105/joss.06889
# https://scores.readthedocs.io/en/2.0.0/tutorials/Fractions_Skill_Score.html

# pip install scores
from scores.spatial import fss_2d_single_field
from scores.fast.fss.typing import FssComputeMethod

def CalculateFSS_scores(forecast,observation,threshold,window_size):
    compute_method = FssComputeMethod.NUMPY
    threshold_operator = np.greater_equal
    # threshold_operator = np.greater
    
    fs_score = fss_2d_single_field(
        forecast,
        observation,
        event_threshold=threshold,
        window_size=window_size,           # same interpretation as 'scale'
        threshold_operator=threshold_operator,
        compute_method=compute_method # default and fastest
    )
    return fs_score*100

In [ ]:
##Run_FSS

scale=5
def Run_FSS(forecast,observation, thresholds=[0,20,40,65], printstatement=False):
    #scale: nxn pixels
    fs_scores = []
    for threshold in thresholds:
        # fs_score = CalculateFSS_pysteps(forecast=forecast,observation=observation,threshold=threshold,scale=scale)
        # if printstatement==True:
            # print(f"FSS = {fs_score:.2f}% for threshold = {threshold} dBZ")
        
        fs_score = CalculateFSS_scores(forecast=forecast,observation=observation,threshold=threshold,window_size=(scale,scale))
        if printstatement==True:
            print(f"FSS = {fs_score:.2f}% for threshold = {threshold} dBZ")
        fs_scores.append(fs_score)
    return fs_scores, thresholds

In [ ]:
##########################
#LOADING DATA
running = True #keep true when using job array
running = False

In [ ]:
##########################
#CALCULATING FOR ALL TIMESTEPS

In [ ]:
def GetFilePath(ModelData, t):
    """
    Build the FSS filename using ModelData and load the .pkl file.
    Creates output directory if needed.
    Returns (fullFilePath, loadedData or None).
    """

    # Build file name
    fileName = (
        f"FractionSkillScore_3D_{ModelData.region}_"
        f"{ModelData.case}_spinup{ModelData.spinup_hours}hrs_{ModelData.timeStrings[t]}.pkl"
    )

    # Build directory for FSS output
    outputDir = os.path.join(
        DirectoryManager.GetOutputDirectory(codeType, dataType),
        "FractionSkillScore",
        f"{ModelData.region}_{ModelData.case}_spinup{ModelData.spinup_hours}hrs",
    )
    os.makedirs(outputDir, exist_ok=True)

    # Full path to the .pkl file
    fullFilePath = os.path.join(outputDir, fileName)

    # No cached file found
    return fullFilePath


In [ ]:
def RunCode():   
    for t in tqdm(loop_elements):
        fullFilePath = GetFilePath(ModelData_NSSL, t)
    
        modelRadarData_NSSL, modelRadarData_TEMPO, radarData_MRMS = GetData(t)
    
        # Per-height storage
        fss_levels_NSSL = []
        fss_levels_TEMPO = []

        nz = radarData_MRMS.sizes["heightAboveSea"]
        for k in range(nz):
    
            forecast_1 = modelRadarData_NSSL.isel(nVertLevels=k).data
            forecast_2 = modelRadarData_TEMPO.isel(nVertLevels=k).data
            observation = radarData_MRMS.isel(heightAboveSea=k).data
    
            fs1, thresholds = Run_FSS(forecast_1, observation, printstatement=False)
            fs2, _ = Run_FSS(forecast_2, observation, printstatement=False)
    
            fss_levels_NSSL.append(fs1)
            fss_levels_TEMPO.append(fs2)
    
        # -------------------------------------------------------
        # 3. Save newly computed FSS results
        # -------------------------------------------------------
        data_to_save = {
            "FSS_NSSL":  np.array(fss_levels_NSSL),
            "FSS_TEMPO": np.array(fss_levels_TEMPO),
            "thresholds": thresholds,
            "zlevels": modelRadarData_NSSL.nVertLevels,
            "timestep": t,
        }
    
        with open(fullFilePath, "wb") as f:
            pickle.dump(data_to_save, f)
    
        print(f"Saved FSS results to {fullFilePath}")

In [ ]:
if running:
    RunCode()

In [ ]:
####################################
#RECOMBINING
recombining = False #keep false when job_array is running
recombining = True

In [ ]:
def Recombine():
    """
    Recombine per-timestep FSS pickle files into full time series.
    """

    FSS_NSSL_all  = []
    FSS_TEMPO_all = []

    thresholds = None
    zlevels    = None

    for t in tqdm(range(ModelData_NSSL.Ntime), desc="Recombining FSS"):

        filePath = GetFilePath(ModelData_NSSL, t)

        # Skip missing timesteps (important for job arrays)
        if not os.path.exists(filePath):
            continue
            
        with open(filePath, "rb") as f:
            data = pickle.load(f)

        FSS_NSSL_all.append(data["FSS_NSSL"])
        FSS_TEMPO_all.append(data["FSS_TEMPO"])

        # Store metadata once
        if thresholds is None:
            thresholds = data["thresholds"]
            zlevels    = data["zlevels"]

    scores_array_NSSL  = np.array(FSS_NSSL_all)
    scores_array_TEMPO = np.array(FSS_TEMPO_all)

    return scores_array_NSSL,scores_array_TEMPO,thresholds,zlevels

In [ ]:
if recombining:
    [scores_array_NSSL_tz,scores_array_TEMPO_tz,thresholds,zlevels] = Recombine()

In [ ]:
###################
#PLOTTING FUNCTIONS
plotting = False #keep false when running job array
plotting = True

In [ ]:
#HELPER FUNCTIONS
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())

In [ ]:
def MakeFSSPlot_Contour(scores_array_NSSL_tz, scores_array_TEMPO_tz):

    fig = plt.figure(figsize=(12, 5))
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05], wspace=0.1)  # <- reduced wspace
    
    cmap = 'turbo'
    
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharey=ax1)
    ax2.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
    cax = fig.add_subplot(gs[2])
    pos = cax.get_position()  # Bbox: [x0, y0, width, height]
    cax.set_position([pos.x0 - 0.01, pos.y0, pos.width, pos.height])
    
    # Data
    a1 = scores_array_NSSL_tz[:, :, 0]
    a2 = scores_array_TEMPO_tz[:, :, 0]
    
    # Color limits
    vmin = 0
    vmax = 100
    
    # Levels for smooth shading
    levels = np.linspace(vmin, vmax, 20)
    
    # Example time and height axes (replace with your real ones)
    time_strings = ModelData_NSSL.timeStrings
    times = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]
    RadarObservationLevels = RadarObservationMask_Class.LoadRadarObservationLevels_MRMS(
        DirectoryManager, ModelData_NSSL
    )
    
    # Contour plots
    cf1 = ax1.contourf(times, RadarObservationLevels, a1.T, 
                       levels=levels, vmin=vmin, vmax=vmax, cmap=cmap)
    cf2 = ax2.contourf(times, RadarObservationLevels, a2.T, 
                       levels=levels, vmin=vmin, vmax=vmax, cmap=cmap)
    
    ax1.set_title("NSSL", fontsize=fontSettings['labelFont'])
    ax2.set_title("TEMPO", fontsize=fontSettings['labelFont'])
    ax1.set_ylabel("Altitude (km)", fontsize=fontSettings['labelFont'])

    ax1.tick_params(axis="both", labelsize=fontSettings["tickFont"]-2)
    ax2.tick_params(axis="both", labelsize=fontSettings["tickFont"]-2)
    
    # Rotate x-ticks 45°
    for ax in [ax1, ax2]:
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Shared colorbar with ticks every 10
    cbar = fig.colorbar(cf1, cax=cax)
    cbar.set_label("FSS (%)", rotation=90, labelpad=0, fontsize=fontSettings['labelFont'])
    cbar.set_ticks(np.arange(vmin, vmax + 1, 10))
    cbar.ax.tick_params(axis="both", labelsize=fontSettings["tickFont"])
    
    # YLim
    for ax in [ax1, ax2]:
        ax.set_ylim(bottom=0, top=16)
        ax.set_xlabel("Time", fontsize=fontSettings['labelFont'])
    

    fig.suptitle(
        f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
        fontsize=fontSettings['titleFont'],
        y=0.96,
        fontweight="bold",
    )
    return fig

In [ ]:
def MakeFSSPlot_Line(scores_array_NSSL, scores_array_TEMPO, thresholds):
    """
    Plot FSS time series for NSSL (solid) and TEMPO (dashed)
    with two legends: thresholds and model line styles.
    """

    #time axis
    time_strings = ModelData_NSSL.timeStrings
    times = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

    # Colors for thresholds
    colors = [
        "black",   # threshold 0
        "blue",    # threshold 1
        "purple",  # threshold 2
        # "red",     # threshold 3 #skipping last threshold
    ]

    # Create figure & axis
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # ------------------------------------------------------
    # Plotting lines
    # ------------------------------------------------------
    for i in range(scores_array_NSSL.shape[1] - 1 ): #skipping last threshold
        ax.plot(
            times,
            scores_array_NSSL[:, i],
            linestyle='solid',
            color=colors[i],
            label=f"{thresholds[i]} dBZ"
        )

        ax.plot(
            times,
            scores_array_TEMPO[:, i],
            linestyle='dashed',
            color=colors[i]
        )

    # Axis limits (flush last tick to edge)
    
    SetXLimitsDatetime(ax, time_array=times)
    fig.autofmt_xdate(rotation=45)
    ax.set_ylim(bottom=0)

    # ------------------------------------------------------
    # Legend 1: Threshold colors (left)
    # ------------------------------------------------------
    threshold_legend = [
        Line2D(
            [0], [0],
            color=colors[i],
            linestyle='solid',
            linewidth=2,
            label=f"{thresholds[i]} dBZ"
        )
        for i in range(len(thresholds) - 1) #skipping last threshold
    ]

    leg1 = ax.legend(
        handles=threshold_legend,
        title="Thresholds",
        loc="upper left",
        fontsize=fontSettings["legendFont"]
    )
    leg1.get_title().set_fontsize(fontSettings["legendFont"]) 

    # ------------------------------------------------------
    # Legend 2: Model styles (top center)
    # ------------------------------------------------------
    model_legend = [
        Line2D([0], [0], color="black", linestyle='solid', linewidth=2, label="NSSL"),
        Line2D([0], [0], color="black", linestyle='dashed', linewidth=2, label="TEMPO"),
    ]

    leg2 = ax.legend(
        handles=model_legend,
        title=f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
        loc="lower center",
        bbox_to_anchor=(0.5, 0.97),  # anchor JUST above the axes, not inside it
        ncol=2,
        frameon=False,
        fontsize=fontSettings["legendFont"]
    )
    leg2.get_title().set_fontsize(fontSettings["titleFont"])
    leg2.get_title().set_fontweight('bold')

    # Add first legend back so both show
    ax.add_artist(leg1)

    # ------------------------------------------------------
    # Labels, grid, title
    # ------------------------------------------------------
    # ax.set_xlabel("Time") 
    ax.set_ylabel("FSS (%)", fontsize=fontSettings['labelFont'])
    ax.set_xlabel("Time", fontsize=fontSettings['labelFont'])
    # ax.set_title(f"Plot of Fraction Skill Score: window = ({scale},{scale})")
    ax.grid(True)

    ax.tick_params(axis="both", labelsize=fontSettings["tickFont"])
    
    return fig

In [ ]:
def GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory):
    outputSubDirectory = f"{ModelData_1.region}_{ModelData_1.case}_{ModelData_1.spinup_hours}hrs"
    
    outputFilePath = os.path.join(
        outputPlottingDirectory,
        "FractionSkillScore",
        outputSubDirectory)
    os.makedirs(outputFilePath, exist_ok=True)
    return outputFilePath

def SaveFigure(fig, ModelData_1,ModelData_2,
               plotType="contour"):
    """
    Saves a figure to corresponding directory.
    """
    # --- Define output subdirectory and file path ---
    outputFilePath = GetOutputFile(ModelData_1,ModelData_2, outputPlottingDirectory)
    outputFile = os.path.join(outputFilePath,f"FractionSkillScore_3D_{plotType}_{ModelData_1.mpType}vsMRMSvs{ModelData_2.mpType}.png")

    # --- Save figure ---
    fig.savefig(outputFile, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
#Removing Levels Above maxZLevel km from Timeseries Averages

maxZLevel = 16
def SubsetAltitude(scores_array,
                   maxZLevel=16):
    RadarObservationLevels = RadarObservationMask_Class.LoadRadarObservationLevels_MRMS(DirectoryManager, ModelData_NSSL)
    scores_array[:,np.where(np.array(RadarObservationLevels)>maxZLevel)[0][0]+1:] = np.nan
    return scores_array

if plotting:
    scores_array_NSSL_tz = SubsetAltitude(scores_array_NSSL_tz, maxZLevel)
    scores_array_TEMPO_tz = SubsetAltitude(scores_array_TEMPO_tz, maxZLevel)

In [ ]:
##########################
#PLOTTING

In [ ]:
def GetFSSLine(scores_array_NSSL_tz,scores_array_TEMPO_tz,
               selection_type = "single_level"):
    
    if selection_type == "height_mean":
        scores_array_NSSL = np.nanmean(scores_array_NSSL_tz,axis=1)
        scores_array_TEMPO = np.nanmean(scores_array_TEMPO_tz,axis=1)
    elif selection_type == "single_level":
        km_level=3
        RadarObservationLevels = RadarObservationMask_Class.LoadRadarObservationLevels_MRMS(DirectoryManager, ModelData_NSSL)
        z_idx = np.where(np.array(RadarObservationLevels)==km_level)[0].item()
        scores_array_NSSL = scores_array_NSSL_tz[:,z_idx,:]
        scores_array_TEMPO = scores_array_TEMPO_tz[:,z_idx,:]

    return scores_array_NSSL, scores_array_TEMPO

if plotting:
    [scores_array_NSSL, scores_array_TEMPO] = GetFSSLine(scores_array_NSSL_tz,scores_array_TEMPO_tz)

In [ ]:
if plotting:
    fontSettings = {
        "tickFont": 16+2,
        "labelFont": 18+2,
        "legendFont": 14+2,
        "titleFont": 22,
    }
    fig = MakeFSSPlot_Contour(scores_array_NSSL_tz,scores_array_TEMPO_tz)
    SaveFigure(fig, ModelData_NSSL, ModelData_TEMPO, plotType="contour")

In [ ]:
if plotting:
    fontSettings = {
            "tickFont": 16+2,
            "labelFont": 18+6,
            "legendFont": 14+2,
            "titleFont": 22,
        }
    fig = MakeFSSPlot_Line(scores_array_NSSL, scores_array_TEMPO, thresholds)
    SaveFigure(fig, ModelData_NSSL, ModelData_TEMPO, plotType="line")

In [ ]:
####################################
#PLOTTING ALL SIMULATIONS
plotting = False #keep false when job array is running
plotting = True

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
                      plotType="line",
                      extension="png"):

    inputFilePath = os.path.join(
        outputPlottingDirectory,
        "FractionSkillScore")

    fileName=f"FractionSkillScore_3D_{plotType}_NSSLvsMRMSvsTEMPO" if region != "PRECIP" else f"FractionSkillScore_3D_{plotType}_NSSLvsPRECIPvsTEMPO"

    
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(inputFilePath, inputSubDirectory)
    # --- File path ---
    inputFilePath = os.path.join(
        load_dir,
        f"{fileName}.{extension}"
    )
    return inputFilePath

def GetFilePaths(plotType):
    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        filePaths.append(GetFigureFilePath(region,case,spinup_hours,
                                           plotType))
    return filePaths

In [ ]:
if plotting:
    sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
    from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:
if plotting:
    plotType = "contour"
    filePaths = GetFilePaths(plotType)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 5),
                                                     wspace=0.005,hspace=-0.01,
                                                     dpi=600)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig, saveDirectory=os.path.join(outputPlottingDirectory,"FractionSkillScore"),fileName=f"FractionSkillScore_3D_{plotType}")

In [ ]:
if plotting:
    plotType = "line"
    filePaths = GetFilePaths(plotType="line")
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 5),
                                                     wspace=0.001,hspace=-0.001,
                                                     dpi=900)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig, saveDirectory=os.path.join(outputPlottingDirectory,"FractionSkillScore"),fileName=f"FractionSkillScore_3D_{plotType}")